# 3D.02 — Portfolio Construction

## 1. Research Objective

This notebook converts the validated 12-month TSMOM positions into portfolio weights using Equal Notional Weighting.

Each ETF receives a fixed nominal allocation of 1/16, signed according to its Long/Short/Flat position.

The objective is to validate portfolio weights, gross exposure, and net exposure before economic backtesting.

The Test period (2022–2025) remains untouched.

In [61]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np 

project_root = Path.cwd()

if not (project_root / "src").is_dir():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import importlib
import src.data
import src.signals
import src.strategy

importlib.reload(src.strategy)
importlib.reload(src.data)

from src.research import prepare_momentum_samples
from src.data import download_prices, to_monthly_prices, validate_monthly_prices, prepare_monthly_prices
from src.signals import calculate_momentum, generate_signals, clean_signals_and_targets, generate_targets
from src.strategy import build_tsmom_positions, calculate_portfolio_weights, calculate_portfolio_exposure

## 2. Data & Signal Preparation

This section prepares the monthly ETF price data and reuses the validated TSMOM signal construction pipeline from Phase 3D.01.

- **Universe:** 16 ETFs across four asset classes.
- **Data period:** January 2008 – December 2021.
- **Momentum lookback:** 12 months.
- **Position construction:** One-month lag using `shift(1)`.
- **Missing values:** Undefined signals and positions are preserved as NaN.

The `build_tsmom_positions()` function generates signals and positions without duplicating the underlying calculations.

Temporal alignment is verified before constructing portfolio weights.

In [ ]:
asset_universe = {
    "Equities" : ["SPY", "VGK", "EWJ", "EEM"],
    "Commodities" : ["USO", "DBA", "DBB", "DBC"],
    "Bonds" : ["IEF", "TLT", "LQD", "HYG"],
    "Forex" : ["FXE", "FXY", "FXB", "FXA"]
}

tickers = []

for assets in asset_universe.values():
    tickers.extend(assets)

start_date = "2008-01-01"
end_date = "2022-01-01"
lookback = 12

prices = download_prices(tickers, start_date, end_date, auto_adjust=True, progress=False)

# print(prices.head())

monthly_prices = to_monthly_prices(prices)

# print(monthly_prices.head())

signals, positions = build_tsmom_positions(monthly_prices, lookback=lookback)

print(positions.head())
print(positions.shape)
print(positions.tail())

print(positions.equals(signals.shift(1)))
print(positions.loc["2009-01":"2009-03", ["SPY", "USO"]])

Ticker      DBA  DBB  DBC  EEM  EWJ  FXA  FXB  FXE  FXY  HYG  IEF  LQD  SPY  \
Date                                                                          
2008-01-31  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2008-02-29  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2008-03-31  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2008-04-30  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2008-05-31  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   

Ticker      TLT  USO  VGK  
Date                       
2008-01-31  NaN  NaN  NaN  
2008-02-29  NaN  NaN  NaN  
2008-03-31  NaN  NaN  NaN  
2008-04-30  NaN  NaN  NaN  
2008-05-31  NaN  NaN  NaN  
(168, 16)
Ticker      DBA  DBB  DBC  EEM  EWJ  FXA  FXB  FXE  FXY  HYG  IEF  LQD  SPY  \
Date                                                                          
2021-08-31  1.0  1.0  1.0  1.0  1.0  1.0  1.0 -1.0 -1.0  1.0 -1.0  1.0  1.0   
202

## 3. Portfolio Weight Construction

The strategy uses Equal Notional Weighting across 16 ETFs.

Each ETF receives a fixed notional allocation of 1/16, with the sign determined by its trading position.

- Long: +6.25%
- Short: -6.25%
- Flat: 0%
- Undefined position: NaN

Weights are not redistributed when an ETF becomes Flat.

In [68]:
portfolio_weights = calculate_portfolio_weights(positions)

print(portfolio_weights)
print(portfolio_weights.shape)

gross_exposure = portfolio_weights.abs().sum(axis=1, min_count=1)

print(gross_exposure.loc["2010"].head())

net_exposure = portfolio_weights.sum(axis=1, min_count=1)

print(net_exposure)

Ticker         DBA     DBB     DBC     EEM     EWJ     FXA     FXB     FXE  \
Date                                                                         
2008-01-31     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
2008-02-29     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
2008-03-31     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
2008-04-30     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
2008-05-31     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
...            ...     ...     ...     ...     ...     ...     ...     ...   
2021-08-31  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625 -0.0625   
2021-09-30  0.0625  0.0625  0.0625  0.0625  0.0625 -0.0625  0.0625 -0.0625   
2021-10-31  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625 -0.0625   
2021-11-30  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625 -0.0625   
2021-12-31  0.0625  0.0625  0.0625  0.0625  0.0625 -0.0625 -0.06

## 4. Reusable Functions & Portfolio Exposure Validation

### Reusable Research Infrastructure

During this session, the following reusable functions were implemented:

- `prepare_monthly_prices()` — Data preparation and validation (`src/data.py`).
- `build_tsmom_positions()` — Momentum signals and lagged positions (`src/strategy.py`).
- `calculate_portfolio_weights()` — Equal Notional Weighting (`src/strategy.py`).
- `calculate_portfolio_exposure()` — Gross and Net Exposure (`src/strategy.py`).

Some calculations remain explicitly implemented in this notebook because they were developed before the reusable functions.

The functions have been tested against the expected data structures and selected numerical results.

Future notebooks will reuse these functions to avoid duplicating research logic.

### Portfolio Exposure Validation

The following checks will be performed across the available historical sample:

- Gross Exposure must remain between 0% and 100%.
- Absolute Net Exposure must not exceed Gross Exposure.
- Undefined initial observations must remain NaN.

These checks ensure consistency with the predefined Equal Notional Weighting methodology.

In [ ]:
gross_exposure, net_exposure = calculate_portfolio_exposure(portfolio_weights)

valid_gross_exposure = gross_exposure.dropna().between(0, 1).all()

print(valid_gross_exposure)

valid_net_exposure = (net_exposure.dropna().abs() <= gross_exposure.dropna()).all()

print(valid_net_exposure)

print(gross_exposure.isna().sum())
print(net_exposure.isna().sum())

True
True
13
13


## 5. Conclusion — Portfolio Construction

### Methodology

The portfolio construction framework follows an **Equal Notional Weighting** approach across 16 ETFs representing four asset classes: Equities, Commodities, Bonds, and Forex.

- Each active position receives a fixed absolute target weight of **6.25%** (1/16).
- Long positions receive positive weights; Short positions receive negative weights.
- Flat positions receive zero weight, with no redistribution of unused allocation.
- Portfolio target weights are determined monthly using lagged trading signals to prevent look-ahead bias.

### Reusable Research Infrastructure

The following functions were implemented and tested:

- `prepare_monthly_prices()` in `src/data.py`
- `build_tsmom_positions()` in `src/strategy.py`
- `calculate_portfolio_weights()` in `src/strategy.py`
- `calculate_portfolio_exposure()` in `src/strategy.py`

Some calculations were performed directly in this notebook before being refactored into reusable functions.

### Validation Results

The portfolio construction checks returned the expected results:

| Validation Check | Result |
|---|---|
| Portfolio weights preserve the 16-ETF universe | Passed |
| Gross exposure remains between 0% and 100% | Passed |
| Absolute net exposure does not exceed gross exposure | Passed |
| Initial undefined observations are preserved (13 months) | Passed |
| February 2009 gross exposure equals 100% | Passed |
| February 2009 net exposure equals −62.5% | Passed |

### Limitations and Next Steps

This notebook validates portfolio construction rules and exposure consistency. It does **not** evaluate strategy profitability or risk-adjusted performance.

The weights represent monthly target allocations. Actual rebalancing activity, transaction costs, slippage, and realized portfolio returns have not yet been modeled.

**Next phase — 3E: Realistic Backtesting**

The next phase will evaluate portfolio returns and performance on the Train and Validation periods, incorporate trading frictions, and calculate portfolio-level risk metrics.

The **2022–2025 Test period remains untouched** until Phase 3F.